# Deep Surrogate Modeling for Chemical Reactor Optimization

This notebook runs the complete project: physics model → data generation → deep surrogate → evaluation → optimization → validation → speed comparison.

In [ ]:
# 1. Install/import project
import sys, os
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import *
print("Project root:", ROOT)
print("Inputs:", INPUT_NAMES)
print("Outputs:", OUTPUT_NAMES)

## 2. Physics-based CSTR model

We model the consecutive reaction A → B → C with an exothermic energy balance. The simulator is the ground-truth generator for the surrogate.

In [ ]:
from src.reactor_model import solve_cstr
r = solve_cstr(330.0, 1.2, 2.0, 0.5, 0.25)
print(r)

## 3. Generate the simulation dataset

Latin Hypercube Sampling covers the operating space more uniformly than naive random sampling.

In [ ]:
from src.data_generation import generate_dataset
data = generate_dataset(n_samples=3000)
display(data.head())
print(data.describe())

## 4. Exploratory data analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
df = data
display(df.corr(numeric_only=True).round(2))
sns.pairplot(df[INPUT_NAMES + OUTPUT_NAMES].sample(min(700, len(df)), random_state=42))
plt.show()

## 5. Train the deep surrogate

The model is a multi-output feed-forward neural network. Inputs and outputs are standardized using only the training set.

In [ ]:
from src.surrogate_model import train
model, metrics = train()
display(metrics)

## 6. Evaluate the surrogate

R² close to 1 and low MAE/RMSE indicate accurate approximation over the held-out test set.

In [ ]:
from src.validation import make_plots
make_plots()
from IPython.display import Image, display
for name in OUTPUT_NAMES:
    display(Image(filename=str(RESULTS / f"parity_{name}.png")))

## 7. Surrogate-based optimization

We maximize desired-product concentration C_B using differential evolution over the four physically consistent operating/design variables.

In [ ]:
from src.optimization import run_optimization
x_opt, y_sur, y_phys = run_optimization()
print("x* =", dict(zip(INPUT_NAMES, x_opt)))

## 8. Physics validation

The optimized point is re-simulated with the original nonlinear reactor model. This is the final engineering validation step.

In [ ]:
comparison = __import__("pandas").read_csv(RESULTS / "optimization_validation.csv")
display(comparison)

## 9. Computational speed comparison

We compare repeated surrogate predictions against repeated nonlinear reactor solves. The exact speed-up depends on the machine and software environment.

In [ ]:
from src.validation import speed_test
speed = speed_test(2000)
display(speed)

## 10. Final project summary

**Deliverables:** trained surrogate, quantitative metrics, parity plots, optimization result, physics validation, and speed comparison.

The central engineering message is that the deep model is an approximation—not the source of truth. The final operating point is checked against the physics-based reactor model.